In [ ]:
# Importing Libraries/Packages
import pandas as pd
import numpy as np

### 1. Loading Datasets

In [ ]:
# Reading the file into the dataframe
df = pd.read_csv("Claims_Data_2018_to_2025.csv")

In [ ]:
# Display structure and first rows for intial inspections
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151440 entries, 0 to 151439
Data columns (total 10 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Received Refiled Year    151440 non-null  int64 
 1   Hospital Name            151440 non-null  object
 2   Hospital Classification  151440 non-null  object
 3   Hospital Sector          151440 non-null  object
 4   Hospital Region          151440 non-null  object
 5   Hospital Province        151440 non-null  object
 6   Hospital Municipality    151440 non-null  object
 7   Claim Status             151440 non-null  object
 8   Claims Count             151440 non-null  int64 
 9   Claims Amount            151440 non-null  int64 
dtypes: int64(3), object(7)
memory usage: 11.6+ MB
None
   Received Refiled Year                                      Hospital Name  \
0                   2018    BUENAVISTA PRIMARY CARE FACILITY 1- DOTS CENTER   
1                   2018   

### 2. Clean Column names

In [ ]:
# Standardize column names:
# - remove spaces
# - convert to lowercase
# - replace spaces with underscores
df.columns = df.columns.str.strip() \
                        .str.lower() \
                        .str.replace(" ", "_") \
                        .str.replace("/", "_")

### 3. Data Validation & Standardization

In [ ]:
# List of categorical columns based on MCO1 variables
categorical_cols = [
    'hospital_region',
    'hospital_classification',
    'hospital_sector',
    'claim_status',
    'hospital_province',
    'hospital_municipality'
]

In [ ]:
print(df.columns.tolist())

['received_refiled_year', 'hospital_name', 'hospital_classification', 'hospital_sector', 'hospital_region', 'hospital_province', 'hospital_municipality', 'claim_status', 'claims_count', 'claims_amount']


In [ ]:
# Standardize categorical values:
# - convert to string
# - remove extra spaces
# - convert to uppercase for consistency
for col in categorical_cols:
  df[col] = df[col].astype(str).str.strip().str.upper()

In [ ]:
# Example: Fix the inconsistent values using mapping
region_mapping = {
    'PHRO-6': 'PHRO-VI',    # unifying different naming formats
    'PHRO VI ': 'PHRO VI'
}

status_mapping = {
    'PAID ': 'PAID',
    'DENIED ': 'DENIED',
    'RTH ': 'RTH'
}

In [ ]:
# Applying mapping corrections
df['hospital_region'] = df['hospital_region'].replace(region_mapping)
df['claim_status'] = df['claim_status'].replace(status_mapping)

### 4. Duplicate Detection

In [ ]:
# Identifying duplicate records based on key fields
duplicates = df.duplicated(subset=[
    'received_refiled_year',
    'hospital_name',
    'hospital_region',
    'claim_status',
    'claims_count',
    'claims_amount'
])

In [ ]:
# Display number of duplicates found
print("Duplicate Rows:", duplicates.sum())

Duplicate Rows: 62


In [ ]:
# Remove duplicate records to avoid double counting
df = df.drop_duplicates()

### 5. Data Type Verification

In [ ]:
# Convert year to integer (discrete numeric)
df['received_refiled_year'] = df['received_refiled_year'].astype(int)

In [ ]:
# Convert claims_count to numeric (handles the errors safely)
df['claims_count'] = pd.to_numeric(df['claims_amount'], errors='coerce')

In [ ]:
# Convert claim_amount to numeric (continuous variable)
df['claims_amount'] = pd.to_numeric(df['claims_amount'], errors='coerce')

### 6. Outlier Detection (For IQR methods)

In [ ]:
def detect_outliers(column):
  # Calculate the quartiles
  Q1 = column.quantile(0.25)
  Q3 = column.quantile(0.75)

  # Computes the Interquartile Range (IQR)
  IQR = Q3 - Q1

  # Defines the lower and upper bounds
  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 * IQR

  # Returns the values outside the range
  return column[(column < lower) | (column > upper)]

In [17]:
# Detects the outliers for claims count and amount
outliers_count = detect_outliers(df['claims_count'])
outliers_amount = detect_outliers(df['claims_amount'])

print("Outliers (Claims Count):", len(outliers_count))
print("Outliers (Claims Amount):", len(outliers_amount))

Outliers (Claims Count): 25112
Outliers (Claims Amount): 25112


### 7. Feature Engineering

In [18]:
# Total Claim Volume per Region
group_volume = df.groupby(['hospital_region'])['claims_count'].sum()

In [19]:
# Total Financial Volumer per Year
financial_volume = df.groupby(['received_refiled_year'])['claims_amount'].sum()

In [20]:
# Claim Status Percentage
status_pct = df.groupby(['hospital_region', 'claim_status'])['claims_count'].sum()

In [21]:
# Converts count into percentages within each region
status_pct = status_pct.groupby(level=0).apply(lambda x: 100 * x / x.sum())

In [23]:
# Denial Rate and RTH Rate
total_claims = df.groupby('hospital_region')['claims_count'].sum()

In [25]:
# Filters the denied and RTH Claims
denied = df[df['claim_status'] == 'DENIED'].groupby('hospital_region')['claims_count'].sum()
rth = df[df['claim_status'] == 'RTH'].groupby('hospital_region')['claims_count'].sum()

In [26]:
# Computes the rates
denial_rate = (denied / total_claims) * 100
rth_rate = (rth / total_claims) * 100

In [27]:
# Average Claim Amount
# Computation for average value per claim
df['avg_claim_amount'] = df['claims_amount'] / df['claims_count']

### 8. Save Clean Dataset

In [29]:
# Exports the cleaned datasets for further analysis or visualization
df.to_csv("cleaned_claims_data.csv", index=False)